# Notebook 2 — Geospatial Integration & Feature Engineering
**Deadline:** Apr 5 (Weeks 7–10)  
**Goal:** Spatial mapping of 25km GEFS grid → county centroids, temporal alignment with 1-hr lag, compound feature engineering. Output: master training DataFrame.

---

**Key methodology references:**
- Hill et al. (2023): Spatial neighborhood (40km radius) + GEFS ingredients
- Alpay et al. (2020): 1-hour lag between peak wind and outage report
- Saki et al. (2025): Compound hazard features (wind × soil moisture)
- Tervo et al. (2021): Static land-use / forest fraction feature

In [1]:
# ============================================================
# IMPORTS
# ============================================================
import os
import json
import warnings
from pathlib import Path
import gcsfs
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
from scipy.spatial import cKDTree
from scipy.ndimage import uniform_filter
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from tqdm import tqdm

warnings.filterwarnings("ignore")

# --- Config ---
PROJECT_ROOT  = Path("~/Capstone2026/Testingthings").expanduser()
DATA_RAW      = PROJECT_ROOT / "data" / "raw"
DATA_PROC     = PROJECT_ROOT / "data" / "processed"
GEFS_DIR      = DATA_RAW / "gefs"
OUTPUT_DIR    = PROJECT_ROOT / "outputs"
DATA_PROC.mkdir(parents=True, exist_ok=True)

SPATIAL_RADIUS_KM = 40
TEMPORAL_LAG_HR   = 1
OUTAGE_THRESHOLD  = 0.01

# Domain bounds — FEMA Region 5
DOMAIN_LON = (-97.0, -80.0)
DOMAIN_LAT = (36.5,   47.5)

# Zarr URLs (from Notebook 1)
GEFS_ZARR_URL = "https://data.dynamical.org/noaa/gefs/analysis/latest.zarr"
ERA5_AR_URL = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"

print("Imports OK.")

Imports OK.


---
## Part 1 — Load Staged Data

In [2]:
print(DATA_RAW)


/data/keeling/a/tob3/Capstone2026/Testingthings/data/raw


In [3]:
# Load EAGLE-I (from Notebook 1)
eaglei = pd.read_csv(DATA_RAW / "eaglei_region5_all.csv", parse_dates=["run_start_time"], low_memory=False)
print(f"EAGLE-I loaded: {len(eaglei):,} records")
print(eaglei.dtypes)
eaglei["run_start_time"] = pd.to_datetime(eaglei["run_start_time"])
print(f"EAGLE-I loaded: {len(eaglei):,} records")

# Load county shapefile
counties = gpd.read_file(DATA_RAW / "counties" / "counties_region5.gpkg")
counties = counties.to_crs(epsg=4326)
print(f"Counties loaded: {len(counties)} counties")
counties[["STATEFP", "COUNTYFP", "NAME", "centroid_lon", "centroid_lat"]].head()

EAGLE-I loaded: 11,998,643 records
fips_code                  int64
county                       str
state                        str
customers_out            float64
run_start_time    datetime64[us]
fips_str                   int64
state_fips                 int64
outage_event               int64
dtype: object
EAGLE-I loaded: 11,998,643 records
Counties loaded: 524 counties


,STATEFP,COUNTYFP,NAME,centroid_lon,centroid_lat
0,39,063,Hancock,-83.666532,41.001936
1,39,003,Allen,-84.105776,40.771518
2,55,111,Sauk,-89.948216,43.426665
3,39,085,Lake,-81.250938,41.910351
4,26,109,Menominee,-87.509684,45.525122


In [4]:
# ============================================================
# OPEN ZARR STORES — lazy, no data downloaded yet
# ============================================================

# GEFS via dynamical.org (free, no credentials)
print("Opening GEFS Zarr...")
ds_gefs_full = xr.open_zarr(GEFS_ZARR_URL, chunks="auto")
GEFS_VARS = ["wind_u_10m", "wind_v_10m", "wind_u_100m", "wind_v_100m",
             "precipitation_surface", "precipitable_water_atmosphere",
             "relative_humidity_2m"]
ds_gefs_r5 = ds_gefs_full[[v for v in GEFS_VARS if v in ds_gefs_full]].sel(
    latitude=slice(DOMAIN_LAT[1], DOMAIN_LAT[0]),
    longitude=slice(DOMAIN_LON[0], DOMAIN_LON[1]),
)
gefs_lons = ds_gefs_r5.longitude.values
gefs_lats = ds_gefs_r5.latitude.values
print(f"GEFS Region 5: {dict(ds_gefs_r5.dims)}")

# ============================================================
# ERA5 from TWO ARCO stores (different variables in each)
# CO store: CAPE, soil moisture
# AR store: wind gusts
# ============================================================

print("Opening ERA5 CO Zarr (CAPE, soil moisture)...")
fs_gcs = gcsfs.GCSFileSystem(token="anon")
ERA5_CO_URL = "gs://gcp-public-data-arco-era5/co/single-level-reanalysis.zarr-v2"
ds_era5_co_full = xr.open_zarr(fs_gcs.get_mapper(ERA5_CO_URL), chunks="auto")

# CO store uses flattened grid
ERA5_CO_VARS = ["cape", "swvl1"]
ERA5_CO_AVAIL = [v for v in ERA5_CO_VARS if v in ds_era5_co_full]

era5_co_lons_all = ds_era5_co_full.longitude.values
era5_co_lats_all = ds_era5_co_full.latitude.values

era5_lon_w = DOMAIN_LON[0] % 360
era5_lon_e = DOMAIN_LON[1] % 360

val_idx_co = np.where(
    (era5_co_lons_all >= era5_lon_w) & (era5_co_lons_all <= era5_lon_e) &
    (era5_co_lats_all >= DOMAIN_LAT[0]) & (era5_co_lats_all <= DOMAIN_LAT[1])
)[0]

ds_era5_co = ds_era5_co_full[ERA5_CO_AVAIL].isel(values=val_idx_co)
era5_co_lons = ds_era5_co.longitude.values
era5_co_lats = ds_era5_co.latitude.values
print(f"ERA5 CO Region 5: {len(val_idx_co):,} points | vars: {ERA5_CO_AVAIL}")

# AR store for wind gusts
print("Opening ERA5 AR Zarr (wind gusts)...")
ERA5_AR_URL = "gs://gcp-public-data-arco-era5/ar/full_37-1h-0p25deg-chunk-1.zarr-v3"
ds_era5_ar_full = xr.open_zarr(fs_gcs.get_mapper(ERA5_AR_URL), chunks="auto")
ERA5_AR_VARS = ["10m_wind_gust_since_previous_post_processing"]
ERA5_AR_AVAIL = [v for v in ERA5_AR_VARS if v in ds_era5_ar_full]


# AR store uses 0-360 longitude, convert domain bounds
era5_ar_lon_w = DOMAIN_LON[0] % 360  # -97 → 263
era5_ar_lon_e = DOMAIN_LON[1] % 360  # -80 → 280

ds_era5_ar = ds_era5_ar_full[ERA5_AR_AVAIL].sel(
    latitude=slice(DOMAIN_LAT[1], DOMAIN_LAT[0]),
    longitude=slice(DOMAIN_LON[0] % 360, DOMAIN_LON[1] % 360)
)
era5_ar_lons = ds_era5_ar.longitude.values
era5_ar_lats = ds_era5_ar.latitude.values
print(f"ERA5 AR Region 5: lat={len(era5_ar_lats)}, lon={len(era5_ar_lons)} | vars: {ERA5_AR_AVAIL}")

print("\nCombined ERA5 variables:", ERA5_CO_AVAIL + ERA5_AR_AVAIL)

Opening GEFS Zarr...
GEFS Region 5: {'time': 76853, 'latitude': 45, 'longitude': 69}
Opening ERA5 CO Zarr (CAPE, soil moisture)...
ERA5 CO Region 5: 1,827 points | vars: ['cape', 'swvl1']
Opening ERA5 AR Zarr (wind gusts)...
ERA5 AR Region 5: lat=45, lon=69 | vars: ['10m_wind_gust_since_previous_post_processing']

Combined ERA5 variables: ['cape', 'swvl1', '10m_wind_gust_since_previous_post_processing']


In [5]:
# ============================================================
# EXTRACT COORDINATE ARRAYS FOR SPATIAL INDEX
# ============================================================

print("Extracting coordinate arrays...")

# GEFS coordinates
gefs_lons = ds_gefs_r5.longitude.values
gefs_lats = ds_gefs_r5.latitude.values
print(f"GEFS grid: {len(gefs_lons)} lons × {len(gefs_lats)} lats")

# ERA5 CO coordinates (flattened 1D grid)
era5_co_lons = ds_era5_co.longitude.values
era5_co_lats = ds_era5_co.latitude.values
print(f"ERA5 CO: {len(era5_co_lons)} points (flattened)")

# ERA5 AR coordinates (regular 2D grid)
era5_ar_lons = ds_era5_ar.longitude.values
era5_ar_lats = ds_era5_ar.latitude.values
print(f"ERA5 AR grid: {len(era5_ar_lons)} lons × {len(era5_ar_lats)} lats")

print("Coordinates extracted.")

Extracting coordinate arrays...
GEFS grid: 69 lons × 45 lats
ERA5 CO: 1827 points (flattened)
ERA5 AR grid: 69 lons × 45 lats
Coordinates extracted.


In [6]:
print("\n" + "="*60)
print("ERA5 DIAGNOSTIC:")
print(f"CO variables requested: {ERA5_CO_VARS}")
print(f"CO variables found: {ERA5_CO_AVAIL}")
print(f"AR variables requested: {ERA5_AR_VARS}")
print(f"AR variables found: {ERA5_AR_AVAIL}")

# Test CO store
if 'cape' in ds_era5_co:
    test_sample = ds_era5_co['cape'].sel(time="2021-06-01T00", method="nearest").values
    print(f"CAPE sample: min={np.nanmin(test_sample):.2f}, max={np.nanmax(test_sample):.2f} J/kg")
    
if 'swvl1' in ds_era5_co:
    test_sample = ds_era5_co['swvl1'].sel(time="2021-06-01T00", method="nearest").values
    print(f"Soil moisture sample: min={np.nanmin(test_sample):.4f}, max={np.nanmax(test_sample):.4f} m3/m3")

# Test AR store  
if '10m_wind_gust_since_previous_post_processing' in ds_era5_ar:
    test_sample = ds_era5_ar['10m_wind_gust_since_previous_post_processing'].sel(time="2021-06-01T00", method="nearest").values
    print(f"Wind gust sample: min={np.nanmin(test_sample):.2f}, max={np.nanmax(test_sample):.2f} m/s")
    
print("="*60)


ERA5 DIAGNOSTIC:
CO variables requested: ['cape', 'swvl1']
CO variables found: ['cape', 'swvl1']
AR variables requested: ['10m_wind_gust_since_previous_post_processing']
AR variables found: ['10m_wind_gust_since_previous_post_processing']
CAPE sample: min=0.00, max=1402.25 J/kg
Soil moisture sample: min=-0.0000, max=0.6275 m3/m3
Wind gust sample: min=1.17, max=12.28 m/s


---
## Part 2 — Spatial Neighborhood Mapping (Hill et al. 2023)

Map the 0.25-degree (~25km) GEFS grid to county centroids using a **40km radius spatial average**.
This smooths small-scale displacement errors inherent in ensemble models.

In [25]:
# ============================================================
# SPATIAL NEIGHBORHOOD MAPPER
# For each county centroid, find all GEFS grid points within
# SPATIAL_RADIUS_KM and pre-compute their indices.
# This index is built once and reused for all dates/variables.
# ============================================================

def haversine_km(lon1, lat1, lon2, lat2):
    """Vectorized haversine distance in km."""
    R = 6371.0
    dlon = np.radians(lon2 - lon1)
    dlat = np.radians(lat2 - lat1)
    a = np.sin(dlat/2)**2 + np.cos(np.radians(lat1)) * np.cos(np.radians(lat2)) * np.sin(dlon/2)**2
    return R * 2 * np.arcsin(np.sqrt(a))


def build_spatial_index(gefs_lons, gefs_lats, county_centroids_df, radius_km=40):
    """
    Build a mapping: county FIPS → list of (flat_idx) GEFS grid points within radius_km.
    
    Uses a kd-tree for efficiency.
    """
    # GEFS grid as flat arrays
    lon_grid, lat_grid = np.meshgrid(gefs_lons, gefs_lats)
    lon_flat = lon_grid.flatten()
    lat_flat = lat_grid.flatten()

    

    # Convert to radians for kd-tree on unit sphere
    def to_xyz(lon_deg, lat_deg):
        lon_r = np.radians(lon_deg)
        lat_r = np.radians(lat_deg)
        return np.column_stack([
            np.cos(lat_r) * np.cos(lon_r),
            np.cos(lat_r) * np.sin(lon_r),
            np.sin(lat_r),
        ])

    tree = cKDTree(to_xyz(lon_flat, lat_flat))
    chord = 2 * np.sin(np.radians(radius_km / 111.0) / 2)   # approx

    spatial_index = {}
    for _, row in county_centroids_df.iterrows():
        fips = row["STATEFP"] + row["COUNTYFP"]
        xyz  = to_xyz(np.array([row["centroid_lon"]]), np.array([row["centroid_lat"]]))
        idxs = tree.query_ball_point(xyz[0], chord)
        spatial_index[fips] = np.array(idxs, dtype=int)

    n_matched = [len(v) for v in spatial_index.values()]
    print(f"Spatial index built for {len(spatial_index)} counties.")
    print(f"Grid points per county: min={min(n_matched)}, mean={np.mean(n_matched):.1f}, max={max(n_matched)}")
    return spatial_index, lon_flat, lat_flat


print("Spatial mapper defined. Will build index when first GEFS file is loaded.")

Spatial mapper defined. Will build index when first GEFS file is loaded.


In [26]:
# ============================================================
# EXTRACT COUNTY-LEVEL FEATURES FROM A GEFS DATASET
# ============================================================

def extract_county_features(ds_dict, spatial_index, lon_flat, lat_flat, stat=("mean", "max")):
    """
    For each variable in ds_dict and each county in spatial_index,
    compute the mean and max over the 40km neighborhood.

    Returns a DataFrame with one row per county.
    """
    records = []

    for fips, grid_idxs in spatial_index.items():
        row = {"fips": fips}
        if len(grid_idxs) == 0:
            continue
        for var_name, ds in ds_dict.items():
            # Get the primary data variable
            data_var = [v for v in ds.data_vars][0]
            arr = ds[data_var].values.flatten()   # flatten spatial dims
            if len(arr) < max(grid_idxs) + 1:
                continue
            neighborhood = arr[grid_idxs]
            neighborhood = neighborhood[~np.isnan(neighborhood)]
            if len(neighborhood) == 0:
                continue
            if "mean" in stat:
                row[f"{var_name}_mean"] = float(np.mean(neighborhood))
            if "max" in stat:
                row[f"{var_name}_max"] = float(np.max(neighborhood))
        records.append(row)

    return pd.DataFrame(records).set_index("fips")


print("Feature extractor defined.")

Feature extractor defined.


---
## Part 3 — Feature Engineering

Compute compound/derived features as proposed by Saki et al. (2025) and Tervo et al. (2021).

In [27]:
# ============================================================
# COMPOUND FEATURE ENGINEERING  (Saki et al. 2025)
# ============================================================

def engineer_compound_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add compound / interaction features to the feature DataFrame.
    These capture the amplification of outage risk when multiple
    hazards co-occur (Saki et al. 2025 hypothesis).
    """
    df = df.copy()

    # Wind shear magnitude (from U + V components)
    if "ugrd_shear_mean" in df and "vgrd_shear_mean" in df:
        df["wind_shear_mag_mean"] = np.sqrt(
            df["ugrd_shear_mean"]**2 + df["vgrd_shear_mean"]**2
        )
        df["wind_shear_mag_max"] = np.sqrt(
            df.get("ugrd_shear_max", df["ugrd_shear_mean"])**2 +
            df.get("vgrd_shear_max", df["vgrd_shear_mean"])**2
        )

    # 10m wind speed magnitude
    if "ugrd_10m_mean" in df and "vgrd_10m_mean" in df:
        df["wind_speed_10m_mean"] = np.sqrt(
            df["ugrd_10m_mean"]**2 + df["vgrd_10m_mean"]**2
        )

    # Wet-soil amplification: gust × soil moisture
    # Hypothesis: same wind speed causes more outages on wet/saturated soil
    if "gust_sfc_max" in df and "soilw_0_10_mean" in df:
        df["wind_gust_x_soilw"] = df["gust_sfc_max"] * df["soilw_0_10_mean"]

    # Rain + wind compound
    if "gust_sfc_max" in df and "apcp_sfc_mean" in df:
        df["wind_gust_x_apcp"] = df["gust_sfc_max"] * df["apcp_sfc_mean"]

    # CAPE × shear: supercell composite proxy
    if "cape_sfc_mean" in df and "wind_shear_mag_mean" in df:
        df["cape_x_shear"] = df["cape_sfc_mean"] * df["wind_shear_mag_mean"]

    return df


print("Compound feature engineering function defined.")

Compound feature engineering function defined.


In [28]:
# ============================================================
# STATIC FEATURES — real data
# Pop density: Census 2020 county estimates (free CSV)
# Forest fraction: USDA FIA county forest stats (free CSV)
# ============================================================

def build_static_features(counties_gdf: gpd.GeoDataFrame) -> pd.DataFrame:
    import requests, io

    static = counties_gdf[["STATEFP", "COUNTYFP", "ALAND"]].copy()
    static["fips"] = static["STATEFP"] + static["COUNTYFP"]
    static = static.set_index("fips")
    static["land_area_km2"] = static["ALAND"] / 1e6

    # --- Population density from Census 2020 estimates ---
    print("Downloading Census population estimates...")
    pop_url = (
        "https://www2.census.gov/programs-surveys/popest/datasets/"
        "2020-2023/counties/totals/co-est2023-alldata.csv"
    )
    pop_df = pd.read_csv(pop_url, encoding="latin-1", dtype={"STATE": str, "COUNTY": str})
    pop_df["fips"] = pop_df["STATE"].str.zfill(2) + pop_df["COUNTY"].str.zfill(3)
    # Remove state-level rows (COUNTY == "000")
    pop_df = pop_df[pop_df["COUNTY"] != "000"][["fips", "POPESTIMATE2022"]].copy()
    pop_df = pop_df.set_index("fips").rename(columns={"POPESTIMATE2022": "population"})

    static = static.join(pop_df, how="left")
    static["pop_density"] = static["population"] / static["land_area_km2"].replace(0, np.nan)
    static["pop_density"] = static["pop_density"].fillna(static["pop_density"].median())
    print(f"  Pop density: {static['pop_density'].notna().sum()}/{len(static)} counties filled")

    # --- Forest fraction from USDA Forest Service (county-level canopy cover) ---
    print("Downloading USFS county canopy cover...")
    try:
        canopy_url = (
            "https://apps.fs.usda.gov/fia/datamart/CSV/datamart_conus.zip"
        )
        canopy_df = pd.read_csv(
            canopy_url,
            compression="zip",
            dtype={"STATECD": str, "COUNTYCD": str},
            usecols=["STATECD", "COUNTYCD", "FOREST_AREA", "TOTAL_LAND_AREA"]
        )
        canopy_df["fips"] = canopy_df["STATECD"].str.zfill(2) + canopy_df["COUNTYCD"].str.zfill(3)
        canopy_df["forest_fraction"] = (
            canopy_df["FOREST_AREA"] / canopy_df["TOTAL_LAND_AREA"]
        ).clip(0, 1)
        canopy_df = canopy_df[["fips", "forest_fraction"]].set_index("fips")
        static = static.join(canopy_df, how="left")
        n_filled = static["forest_fraction"].notna().sum()
        print(f"  Forest fraction: {n_filled}/{len(static)} counties filled")
        if n_filled < len(static) * 0.5:
            raise ValueError("Too few counties matched")
    except Exception as e:
        print(f"  USFS download failed ({e}) — using state-level proxy")
        state_forest = {
            "17": 0.14,  # IL
            "18": 0.22,  # IN
            "26": 0.55,  # MI
            "27": 0.38,  # MN
            "39": 0.28,  # OH
            "55": 0.46,  # WI
        }
        static["forest_fraction"] = static["STATEFP"].map(state_forest).fillna(0.25)

    static["forest_fraction"] = static["forest_fraction"].fillna(
        static["forest_fraction"].median()
    )

    return static[["land_area_km2", "forest_fraction", "pop_density"]]


static_features = build_static_features(counties)
print(f"\nStatic features shape: {static_features.shape}")
static_features.head()

  Pop density: 524/524 counties filled
  USFS download failed (HTTP Error 404: Not Found) — using state-level proxy

Static features shape: (524, 3)


,land_area_km2,forest_fraction,pop_density
fips,,,
39063,1376.118086,0.28,54.336907
39003,1042.587388,0.28,96.967411
55111,2153.683697,0.46,30.538375
39085,593.807527,0.28,390.254400
26109,2704.053651,0.55,8.588957


---
## Part 4 — Temporal Alignment & Master Training Table

Build the master DataFrame: one row per (county × 3-hour forecast valid time), with GEFS features + EAGLE-I outage label + 1-hour lag applied.

In [29]:
def align_outage_to_gefs_times(eaglei_df: pd.DataFrame, lag_hr: int = 1) -> pd.DataFrame:
    df = eaglei_df.copy()

    # Shift outage time back by lag so weather time aligns with outage
    df["weather_valid_time"] = df["run_start_time"] - pd.Timedelta(hours=lag_hr)

    # Round to nearest 3-hour block
    df["valid_time_3h"] = df["weather_valid_time"].dt.round("3h")

    # Aggregate per county per 3-hour block
    # No total_customers or outage_frac — use raw customers_out and outage_event flag
    agg = (
        df.groupby(["fips_str", "valid_time_3h"])
        .agg(
            customers_out_sum=("customers_out", "sum"),
            customers_out_max=("customers_out", "max"),
            outage_event=("outage_event", "max"),  # 1 if any record in block was an outage
        )
        .reset_index()
        .rename(columns={"fips_str": "fips"})
    )

    return agg


aligned = align_outage_to_gefs_times(eaglei, lag_hr=TEMPORAL_LAG_HR)
print(f"Aligned records: {len(aligned):,}")
print(f"Outage events: {aligned['outage_event'].sum():,} ({aligned['outage_event'].mean()*100:.2f}%)")
aligned.head()

Aligned records: 1,677,118
Outage events: 202,557 (12.08%)


,fips,valid_time_3h,customers_out_sum,customers_out_max,outage_event
0,17001,2021-01-01 00:00:00,333.0,103.0,1
1,17001,2021-01-01 03:00:00,345.0,102.0,1
2,17001,2021-01-01 09:00:00,94.0,11.0,0
3,17001,2021-01-01 12:00:00,886.0,216.0,1
4,17001,2021-01-01 15:00:00,4606.0,587.0,1


In [30]:
# ============================================================
# BUILD ALL THREE SPATIAL INDICES
# ============================================================
print("Building GEFS spatial index...")
gefs_spatial_idx, gefs_lon_flat, gefs_lat_flat = build_spatial_index(
    gefs_lons, gefs_lats, counties, radius_km=SPATIAL_RADIUS_KM
)

def to_xyz(lon_deg, lat_deg):
    lon_r = np.radians(lon_deg)
    lat_r = np.radians(lat_deg)
    return np.column_stack([
        np.cos(lat_r) * np.cos(lon_r),
        np.cos(lat_r) * np.sin(lon_r),
        np.sin(lat_r),
    ])

chord = 2 * np.sin(np.radians(SPATIAL_RADIUS_KM / 111.0) / 2)

print("Building ERA5 CO spatial index...")
era5_co_lons_180 = np.where(era5_co_lons > 180, era5_co_lons - 360, era5_co_lons)
tree_era5_co = cKDTree(to_xyz(era5_co_lons_180, era5_co_lats))
era5_co_spatial_idx = {}
for _, row in counties.iterrows():
    fips = row["STATEFP"] + row["COUNTYFP"]
    xyz = to_xyz(np.array([row["centroid_lon"]]), np.array([row["centroid_lat"]]))
    era5_co_spatial_idx[fips] = np.array(tree_era5_co.query_ball_point(xyz[0], chord), dtype=int)
print(f"ERA5 CO index built for {len(era5_co_spatial_idx)} counties.")

print("Building ERA5 AR spatial index...")
lon_grid_ar, lat_grid_ar = np.meshgrid(era5_ar_lons, era5_ar_lats)
era5_ar_lon_flat = lon_grid_ar.flatten()
era5_ar_lat_flat = lat_grid_ar.flatten()
tree_era5_ar = cKDTree(to_xyz(era5_ar_lon_flat, era5_ar_lat_flat))
era5_ar_spatial_idx = {}
for _, row in counties.iterrows():
    fips = row["STATEFP"] + row["COUNTYFP"]
    xyz = to_xyz(np.array([row["centroid_lon"]]), np.array([row["centroid_lat"]]))
    era5_ar_spatial_idx[fips] = np.array(tree_era5_ar.query_ball_point(xyz[0], chord), dtype=int)
print(f"ERA5 AR index built for {len(era5_ar_spatial_idx)} counties.")

Building GEFS spatial index...
Spatial index built for 524 counties.
Grid points per county: min=0, mean=8.6, max=11
Building ERA5 CO spatial index...
ERA5 CO index built for 524 counties.
Building ERA5 AR spatial index...
ERA5 AR index built for 524 counties.


In [31]:
# ============================================================
# REAL FEATURE EXTRACTOR — one timestamp at a time
# Queries GEFS + ERA5 Zarr stores, applies 40km neighborhood
# average for each county. Called inside build_master_table().
# ============================================================

def extract_features_for_timestamp(ts, ds_gefs_r5, ds_era5_co, ds_era5_ar,
                                    gefs_spatial_idx, era5_co_spatial_idx, era5_ar_spatial_idx):
    """
    Pull one GEFS + ERA5 timestep and extract county neighborhood features.
    Uses TWO ERA5 stores: CO (CAPE, soil) and AR (wind gusts).
    """
    ts = pd.Timestamp(ts)

    # Download Region 5 slice for this timestep
    gefs = ds_gefs_r5.sel(time=ts, method="nearest").compute()
    era5_co = ds_era5_co.sel(time=ts, method="nearest").compute()
    era5_ar = ds_era5_ar.sel(time=ts, method="nearest").compute()

    gefs_flat = {v: gefs[v].values.flatten() for v in gefs.data_vars}
    era5_co_flat = {v: era5_co[v].values.flatten() for v in era5_co.data_vars}
    era5_ar_flat = {v: era5_ar[v].values.flatten() for v in era5_ar.data_vars}

    records = {}
    for fips in gefs_spatial_idx:
        g_idxs = gefs_spatial_idx[fips]
        e_co_idxs = era5_co_spatial_idx.get(fips, g_idxs)
        e_ar_idxs = era5_ar_spatial_idx.get(fips, g_idxs)
        row = {}

        # --- GEFS variables ---
        for zarr_var, col_stem in [
            ("wind_u_10m",                  "ugrd_10m"),
            ("wind_v_10m",                  "vgrd_10m"),
            ("precipitation_surface",       "apcp_sfc"),
            ("precipitable_water_atmosphere","pwat_clm"),
            ("relative_humidity_2m",        "rh_2m"),
        ]:
            if zarr_var not in gefs_flat or len(g_idxs) == 0:
                continue
            nbhd = gefs_flat[zarr_var][g_idxs]
            nbhd = nbhd[~np.isnan(nbhd)]
            if len(nbhd):
                row[f"{col_stem}_mean"] = float(np.mean(nbhd))
                row[f"{col_stem}_max"]  = float(np.max(nbhd))

        # --- Wind shear proxy ---
        if "wind_u_100m" in gefs_flat and "wind_u_10m" in gefs_flat and len(g_idxs):
            u_sh = gefs_flat["wind_u_100m"][g_idxs] - gefs_flat["wind_u_10m"][g_idxs]
            v_sh = gefs_flat["wind_v_100m"][g_idxs] - gefs_flat["wind_v_10m"][g_idxs]
            u_sh = u_sh[~np.isnan(u_sh)]
            v_sh = v_sh[~np.isnan(v_sh)]
            if len(u_sh):
                row["ugrd_shear_mean"] = float(np.mean(u_sh))
                row["vgrd_shear_mean"] = float(np.mean(v_sh))
                row["hlcy_3km_mean"]   = float(np.mean(np.sqrt(u_sh**2 + v_sh**2)))

        # --- ERA5 CO variables (CAPE, soil moisture - flattened grid) ---
        for zarr_var, col_stem in [
            ("cape",  "cape_sfc"),
            ("swvl1", "soilw_0_10"),
        ]:
            if zarr_var not in era5_co_flat or len(e_co_idxs) == 0:
                continue
            nbhd = era5_co_flat[zarr_var][e_co_idxs]
            nbhd = nbhd[~np.isnan(nbhd)]
            if len(nbhd):
                row[f"{col_stem}_mean"] = float(np.mean(nbhd))
                row[f"{col_stem}_max"]  = float(np.max(nbhd))

        # --- ERA5 AR variables (wind gusts - regular grid) ---
        if "10m_wind_gust_since_previous_post_processing" in era5_ar_flat and len(e_ar_idxs):
            nbhd = era5_ar_flat["10m_wind_gust_since_previous_post_processing"][e_ar_idxs]
            nbhd = nbhd[~np.isnan(nbhd)]
            if len(nbhd):
                row["gust_sfc_mean"] = float(np.mean(nbhd))
                row["gust_sfc_max"]  = float(np.max(nbhd))

        records[fips] = row

    return pd.DataFrame.from_dict(records, orient="index")


def build_master_table(aligned_df, static_features_df,
                       ds_gefs_r5, ds_era5_co, ds_era5_ar,
                       gefs_spatial_idx, era5_co_spatial_idx, era5_ar_spatial_idx):
    all_rows = []
    timestamps = sorted(aligned_df["valid_time_3h"].unique())

    for ts in tqdm(timestamps, desc="Building master table"):
        try:
            feats = extract_features_for_timestamp(
                ts, ds_gefs_r5, ds_era5_co, ds_era5_ar, 
                gefs_spatial_idx, era5_co_spatial_idx, era5_ar_spatial_idx
            )
        except Exception as e:
            print(f"  ⚠️  {pd.Timestamp(ts):%Y-%m-%d %H:%M}: {e}")
            continue

        mask   = aligned_df["valid_time_3h"] == ts
        labels = aligned_df[mask].set_index("fips")[["outage_event"]]
        merged = labels.join(feats, how="left")
        merged = merged.join(static_features_df, how="left")
        merged["valid_time_3h"] = ts
        all_rows.append(merged.reset_index())

    if not all_rows:
        print("No rows — check Zarr connections.")
        return pd.DataFrame()

    return pd.concat(all_rows, ignore_index=True)

In [32]:
def build_master_table(aligned_df, static_features_df,
                       ds_gefs_r5, ds_era5_r5,
                       gefs_spatial_idx, era5_spatial_idx,
                       batch_days=90):
    """
    Downloads data in monthly batches instead of one timestamp at a time.
    GEFS and ERA5 downloaded in parallel per batch.
    """
    # At the very top of build_master_table, before anything else:
    aligned_df = aligned_df.copy()
    aligned_df["fips"] = aligned_df["fips"].astype(str).str.zfill(5)

    import concurrent.futures
    all_rows = []
    dates = sorted(aligned_df["valid_time_3h"].dt.date.unique())
    date_batches = [dates[i:i+batch_days] for i in range(0, len(dates), batch_days)]

    for batch in tqdm(date_batches, desc="Building master table (batches)"):
        t_start = pd.Timestamp(batch[0])
        t_end   = pd.Timestamp(batch[-1]) + pd.Timedelta(hours=23)

        try:
            with concurrent.futures.ThreadPoolExecutor(max_workers=2) as ex:
                fut_gefs = ex.submit(lambda: ds_gefs_r5.sel(time=slice(t_start, t_end)).compute())
                fut_era5 = ex.submit(lambda: ds_era5_r5.sel(time=slice(t_start, t_end)).compute())
                gefs_batch = fut_gefs.result()
                era5_batch = fut_era5.result()
        except Exception as e:
            print(f"  ⚠️  Batch {t_start:%Y-%m-%d}: {e}")
            continue

        batch_times = aligned_df[
            (aligned_df["valid_time_3h"].dt.date >= batch[0]) &
            (aligned_df["valid_time_3h"].dt.date <= batch[-1])
        ]["valid_time_3h"].unique()

        for ts in batch_times:
            ts = pd.Timestamp(ts)
            try:
                gefs_t = gefs_batch.sel(time=ts, method="nearest")
                era5_t = era5_batch.sel(time=ts, method="nearest")

                gefs_flat = {v: gefs_t[v].values.flatten() for v in gefs_t.data_vars}
                era5_flat = {v: era5_t[v].values.flatten() for v in era5_t.data_vars}
            except Exception as e:
                print(f"    ⚠️  {ts}: {e}")
                continue

            records = {}
            for fips in gefs_spatial_idx:
                g_idxs = gefs_spatial_idx[fips]
                e_idxs = era5_spatial_idx.get(fips, g_idxs)
                row = {}

                for zarr_var, col_stem in [
                    ("wind_u_10m",                   "ugrd_10m"),
                    ("wind_v_10m",                   "vgrd_10m"),
                    ("precipitation_surface",        "apcp_sfc"),
                    ("precipitable_water_atmosphere","pwat_clm"),
                    ("relative_humidity_2m",         "rh_2m"),
                ]:
                    if zarr_var not in gefs_flat or len(g_idxs) == 0:
                        continue
                    nbhd = gefs_flat[zarr_var][g_idxs]
                    nbhd = nbhd[~np.isnan(nbhd)]
                    if len(nbhd):
                        row[f"{col_stem}_mean"] = float(np.mean(nbhd))
                        row[f"{col_stem}_max"]  = float(np.max(nbhd))

                if "wind_u_100m" in gefs_flat and len(g_idxs):
                    u_sh = gefs_flat["wind_u_100m"][g_idxs] - gefs_flat["wind_u_10m"][g_idxs]
                    v_sh = gefs_flat["wind_v_100m"][g_idxs] - gefs_flat["wind_v_10m"][g_idxs]
                    u_sh = u_sh[~np.isnan(u_sh)]
                    v_sh = v_sh[~np.isnan(v_sh)]
                    if len(u_sh):
                        row["ugrd_shear_mean"] = float(np.mean(u_sh))
                        row["vgrd_shear_mean"] = float(np.mean(v_sh))
                        row["hlcy_3km_mean"]   = float(np.mean(np.sqrt(u_sh**2 + v_sh**2)))

                for zarr_var, col_stem in [
                    ("cape",  "cape_sfc"),
                    ("swvl1", "soilw_0_10"),
                ]:
                    if zarr_var not in era5_flat or len(e_idxs) == 0:
                        continue
                    nbhd = era5_flat[zarr_var][e_idxs]
                    nbhd = nbhd[~np.isnan(nbhd)]
                    if len(nbhd):
                        row[f"{col_stem}_mean"] = float(np.mean(nbhd))
                        row[f"{col_stem}_max"]  = float(np.max(nbhd))

                records[fips] = row

            feats = pd.DataFrame.from_dict(records, orient="index")
            mask   = aligned_df["valid_time_3h"] == ts
            labels = aligned_df[mask].set_index("fips")[["outage_event"]]
            merged = labels.join(feats, how="left")
            merged = merged.join(static_features_df, how="left")
            merged["valid_time_3h"] = ts
            all_rows.append(merged.reset_index())

    if not all_rows:
        print("No rows built — check Zarr connections.")
        return pd.DataFrame()

    return pd.concat(all_rows, ignore_index=True)

print("Batched master table builder defined.")

Batched master table builder defined.


In [33]:
# Save feature column list for use in Notebooks 3 & 4
FEATURE_COLS = [
    "ugrd_10m_mean", "vgrd_10m_mean", "gust_sfc_max", "gust_sfc_mean",
    "ugrd_shear_mean", "vgrd_shear_mean", "cape_sfc_mean", "cape_sfc_max",
    "cin_sfc_mean", "hlcy_3km_mean", "pwat_clm_mean",
    "apcp_sfc_mean", "apcp_sfc_max", "soilw_0_10_mean",
    "wind_shear_mag_mean", "wind_gust_x_soilw", "wind_gust_x_apcp",
    "cape_x_shear", "wind_speed_10m_mean",
    
]

meta = {"feature_cols": FEATURE_COLS, "label_col": "outage_event",
        "data_file": "master_table.parquet"}   # <-- updated from SYNTHETIC
with open(DATA_PROC / "feature_schema.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Feature schema saved.")
print("\n✅ Notebook 2 complete — real data. Ready for 03_modeling.ipynb")

Feature schema saved.

✅ Notebook 2 complete — real data. Ready for 03_modeling.ipynb


All cells below are debugging cells

# Check one timestamp to see what the extractor actually produces
import pandas as pd
ts = pd.Timestamp("2021-06-01 00:00:00")

gefs_t = ds_gefs_r5.sel(time=ts, method="nearest").compute()
era5_t = ds_era5_r5.sel(time=ts, method="nearest").compute()

print("GEFS vars and shapes:")
for v in gefs_t.data_vars:
    print(f"  {v}: {gefs_t[v].values.flatten().shape}")

print("\nERA5 vars and shapes:")
for v in era5_t.data_vars:
    print(f"  {v}: {era5_t[v].values.flatten().shape}")

# Check one county
test_fips = list(gefs_spatial_idx.keys())[0]
g_idxs = gefs_spatial_idx[test_fips]
e_idxs = era5_spatial_idx[test_fips]
print(f"\nCounty {test_fips}:")
print(f"  GEFS idx range: {g_idxs.min()} - {g_idxs.max()}, array size: {gefs_t['wind_u_10m'].values.flatten().shape[0]}")
print(f"  ERA5 idx range: {e_idxs.min()} - {e_idxs.max()}, array size: {era5_t['cape'].values.flatten().shape[0]}")

# Check what values actually come out for one county
test_fips = list(gefs_spatial_idx.keys())[0]
g_idxs = gefs_spatial_idx[test_fips]
e_idxs = era5_spatial_idx[test_fips]

gefs_arr = gefs_t["wind_u_10m"].values.flatten()
era5_arr = era5_t["cape"].values.flatten()

print(f"GEFS wind_u_10m neighborhood values: {gefs_arr[g_idxs]}")
print(f"Any NaN in GEFS neighborhood: {np.isnan(gefs_arr[g_idxs]).any()}")
print(f"\nERA5 cape neighborhood values: {era5_arr[e_idxs]}")
print(f"Any NaN in ERA5 neighborhood: {np.isnan(era5_arr[e_idxs]).any()}")

# Also check the aligned df
print(f"\naligned columns: {aligned.columns.tolist()}")
print(f"aligned shape: {aligned.shape}")
print(aligned.head(3))

# Check fips key format alignment
aligned_fips = set(aligned["fips"].unique())
spatial_fips = set(gefs_spatial_idx.keys())

print(f"Aligned fips sample: {list(aligned_fips)[:5]}")
print(f"Spatial index fips sample: {list(spatial_fips)[:5]}")
print(f"Overlap: {len(aligned_fips & spatial_fips)}")
print(f"In aligned but not spatial: {len(aligned_fips - spatial_fips)}")
print(f"In spatial but not aligned: {len(spatial_fips - aligned_fips)}")